# v6b ReDo fix (GPU 1): continual from phase 60, resetting dead neurons
The continual run from phase 60 (just before it breaks down), same recipe, with one change: at the start of every phase and every 5,000 steps, MLP neurons that fire on fewer than 1% of tokens (checked on the English source and on Hindi, Chinese AND Arabic text, so a neuron used by only one language is not touched) get fresh incoming weights and zero outgoing weights (ReDo, Sokar et al. 2023).
Question: does resetting dead neurons stop the breakdown after phase 60, and does the self survive?
**Just Run All.** Rerunning resumes from the last saved phase. Checkpoints: `runs/v6b_cont_redo_from60/ckpt/`.

In [1]:
import os, sys, glob, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "1"     # run this cell FIRST in a fresh kernel
os.environ["HF_HUB_DISABLE_XET"] = "1"          # HF uploads failed overnight in the new "xet" path; use the classic upload
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common_v6 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
RUN = "v6b_cont_redo_from60"

device: cuda | NVIDIA GeForce RTX 4090


In [2]:
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):                          # retry: HF sometimes drops the connection
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

SRC_RUN, SRC_PHASE = "v4b_continual_s0", 60
if not glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt"):
    name = next(f for f in api.list_repo_files(HF_REPO) if f.startswith(f"{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_"))
    hf_hub_download(HF_REPO, name, local_dir="runs")
INIT = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")[0]
print("branch from:", INIT)

# FLORES dev + devtest
from datasets import load_dataset
for l, code in {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            os.makedirs("data/flores", exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
missing = [p for p in [f"data/en-{l}/{s}.jsonl" for l in LANGS for s in ["train", "val"]] if not os.path.exists(p)]
print("data OK" if not missing else f"MISSING: {missing}")

branch from: runs/v4b_continual_s0/ckpt/p060_ar.pt
data OK


In [3]:
C = RunConfig(run_name=RUN, mode="continual", order=("hi", "zh", "ar"), init_ckpt=INIT, start_phase=SRC_PHASE,
              n_phases=90, seed=0, time_cap_h=120.0, hf_repo=HF_REPO, redo_every=5000, redo_thresh=0.01)
print(C)

RunConfig(run_name='v6b_cont_redo_from60', mode='continual', order=('hi', 'zh', 'ar'), init_ckpt='runs/v4b_continual_s0/ckpt/p060_ar.pt', n_phases=90, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints', start_phase=60, noise_per_step=0.0, boundary_push=0.0, health_every=1, redo_every=5000, redo_thresh=0.01)


In [4]:
D = load_data(train_langs=["hi", "zh", "ar"])

  hi: 993,016 train / 2,000 val
  zh: 755,626 train / 2,000 val
  ar: 998,000 train / 2,000 val
data loaded in 43s


In [ ]:
model = train_run(C, D)

 RUN v6b_cont_redo_from60 | mode continual | 90 phases after phase 0 | device cuda
 phase order: init (loaded) → hi → zh → ar → hi → zh → ar → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
BRANCH: starting from runs/v4b_continual_s0/ckpt/p060_ar.pt (phase 60, ar); next phase 61

▶ PHASE 61/90 | training hi (993,016 pairs)
  ReDo at phase start: reset 2755 MLP neurons that almost never fire


Upload 0 LFS files: 0it [00:00, ?it/s]

[W1028 05:45:25.334802465 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1920991232 bytes (free: 1371275264, total: 25250627584).


  [p61 hi] step    500 | lr 2.0e-04 | train 4.024 | val 2.059 (best 2.059) | window Δ n/a | stable | 75k tok/s | phase 1m38s | run ETA: after first phase | health {'wnorm': 220.899, 'rank_enc1': 676.9, 'rank_dec1': 476.0, 'rarely_active': 0.0459}
  [p61 hi] step  1,000 | lr 2.0e-04 | train 2.950 | val 1.727 (best 1.727) | window Δ n/a | stable | 77k tok/s | phase 3m13s | run ETA: after first phase | health {'wnorm': 221.146, 'rank_enc1': 817.2, 'rank_dec1': 583.9, 'rarely_active': 0.0476}
  [p61 hi] step  1,500 | lr 2.0e-04 | train 2.752 | val 1.592 (best 1.592) | window Δ n/a | stable | 77k tok/s | phase 4m49s | run ETA: after first phase | health {'wnorm': 221.435, 'rank_enc1': 863.1, 'rank_dec1': 624.2, 'rarely_active': 0.0526}
  [p61 hi] step  2,000 | lr 2.0e-04 | train 2.616 | val 1.525 (best 1.525) | window Δ n/a | stable | 75k tok/s | phase 6m26s | run ETA: after first phase | health {'wnorm': 221.752, 'rank_enc1': 917.9, 'rank_dec1': 674.8, 'rarely_active': 0.0515}
  [p61 hi] s

p061_hi.pt:   0%|          | 0.00/267M [00:00<?, ?B/s]

...

No files have been modified since last commit. Skipping to prevent empty commit.


  [p79 hi] step  4,000 | lr 2.0e-04 | train 3.518 | val 2.406 (best 2.406) | window Δ +12.28% | stable | 77k tok/s | phase 12m51s | run ETA ~7h40m | health {'wnorm': 217.399, 'rank_enc1': 120.8, 'rank_dec1': 288.8, 'rarely_active': 0.4037}
  [p79 hi] step  4,500 | lr 2.0e-04 | train 3.500 | val 2.368 (best 2.368) | window Δ +8.95% | stable | 77k tok/s | phase 14m26s | run ETA ~7h40m | health {'wnorm': 217.848, 'rank_enc1': 130.6, 'rank_dec1': 311.3, 'rarely_active': 0.4184}
  ReDo at step 5,000: reset 8825 MLP neurons
  [p79 hi] step  5,000 | lr 2.0e-04 | train 3.488 | val 2.369 (best 2.368) | window Δ +6.27% | stable | 77k tok/s | phase 16m01s | run ETA ~7h40m | health {'wnorm': 213.364, 'rank_enc1': 203.3, 'rank_dec1': 273.5, 'rarely_active': 0.0031}
  [p79 hi] step  5,500 | lr 2.0e-04 | train 3.478 | val 2.353 (best 2.353) | window Δ +4.78% | stable | 75k tok/s | phase 17m38s | run ETA ~7h40m | health {'wnorm': 214.107, 'rank_enc1': 161.8, 'rank_dec1': 345.1, 'rarely_active': 0.1648

p081_ar.pt:   0%|          | 0.00/267M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


  [p82 hi] step    500 | lr 2.0e-04 | train 4.963 | val 3.330 (best 3.330) | window Δ n/a | stable | 75k tok/s | phase 1m37s | run ETA ~5h47m | health {'wnorm': 211.354, 'rank_enc1': 211.9, 'rank_dec1': 149.2, 'rarely_active': 0.0531}
  [p82 hi] step  1,000 | lr 2.0e-04 | train 4.211 | val 3.014 (best 3.014) | window Δ n/a | stable | 77k tok/s | phase 3m12s | run ETA ~5h47m | health {'wnorm': 211.755, 'rank_enc1': 171.9, 'rank_dec1': 174.4, 'rarely_active': 0.1412}
  [p82 hi] step  1,500 | lr 2.0e-04 | train 4.003 | val 2.871 (best 2.871) | window Δ n/a | stable | 77k tok/s | phase 4m47s | run ETA ~5h47m | health {'wnorm': 212.369, 'rank_enc1': 119.3, 'rank_dec1': 216.8, 'rarely_active': 0.2651}
  [p82 hi] step  2,000 | lr 2.0e-04 | train 3.896 | val 2.773 (best 2.773) | window Δ n/a | stable | 75k tok/s | phase 6m24s | run ETA ~5h47m | health {'wnorm': 213.028, 'rank_enc1': 117.9, 'rank_dec1': 236.0, 'rarely_active': 0.3418}
  [p82 hi] step  2,500 | lr 2.0e-04 | train 3.828 | val 2.73

In [ ]:
print_run_summary(RUN)